In [8]:
import pandas as pd

# Flag to switch between datasets
n_50_signal = False

# Define file paths
if n_50_signal:
    EA_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_dual_bounds_and_models\Solve_TSP_time_limit\TSP_EA_dual_bound_verification_results_50_cus_1800s_lim.csv"
    Single_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_dual_bounds_and_models\Solve_TSP_time_limit\TSP_single_dual_bound_50_cus_selected_results_1800s_lim.csv"
    output_filename = 'aggregated_tsp_n50_results.csv'
else:
    EA_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_dual_bounds_and_models\Solve_TSP_time_limit\TSP_EA_dual_bound_verification_results_20_cus_1800s_lim.csv"
    Single_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_dual_bounds_and_models\Solve_TSP_time_limit\TSP_single_dual_bound_20_cus_selected_results_1800s_lim.csv"
    output_filename = 'aggregated_tsp_n20_results.csv'
    
# Read the CSV files
df_ea = pd.read_csv(EA_path)
df_single = pd.read_csv(Single_path)

# --- HELPER FUNCTION: Clean Optimality ---
def clean_optimality(val):
    """
    Converts values starting with 'False' to boolean False.
    Keeps 'True' or boolean True as is.
    """
    s_val = str(val).strip()
    if s_val.lower().startswith('false'):
        return False
    elif s_val.lower().startswith('true'):
        return True
    return val

# Apply the cleaning function to the relevant columns
df_ea['Optimality'] = df_ea['Optimality'].apply(clean_optimality)
df_single['Is Optimal'] = df_single['Is Optimal'].apply(clean_optimality)

# --- 1. Prepare EA DataFrame (File 1) ---
df_ea_clean = df_ea[[
    'Instance', 
    'Objective Value', 
    'Nodes Expanded', 
    'Total Times (s)', 
    'Optimality'
]].rename(columns={
    'Objective Value': 'EA Dual Bound Solution',
    'Nodes Expanded': 'EA Dual Bound Expanded Nodes',
    'Total Times (s)': 'EA Dual Bound Time',
    'Optimality': 'EA Dual Bound Optimality'
})

# --- 2. Prepare Single DataFrame (File 2) ---
df_single_clean = df_single[[
    'Instance', 
    'Cost', 
    'Nodes Expanded', 
    'Running Time (s)', 
    'Is Optimal'
]].rename(columns={
    'Cost': 'Single Dual Bound Solution',
    'Nodes Expanded': 'Single Dual Bound Expanded Nodes',
    'Running Time (s)': 'Single Dual Bound Time',
    'Is Optimal': 'Single Dual Bound Optimality'
})

# --- 3. Merge the DataFrames ---
merged_df = pd.merge(df_ea_clean, df_single_clean, on='Instance', how='outer')

# --- 4. Calculate Best Known Value and Gaps ---
# Assuming these are Dual Bounds (Lower Bounds), we want to MAXIMIZE the value.
# (If they are Costs/Upper Bounds, change .max() to .min())
merged_df['Best Known Solution'] = merged_df[['EA Dual Bound Solution', 'Single Dual Bound Solution']].min(axis=1)

# Calculate Gaps: (Best - Value) / Best
merged_df['EA Dual Bound Optimality Gap'] = (
    (abs(merged_df['Best Known Solution'] - merged_df['EA Dual Bound Solution']) / merged_df['Best Known Solution']) * 100
).map('{:.2f}%'.format)

merged_df['Single Dual Bound Optimality Gap'] = (
    (abs(merged_df['Best Known Solution'] - merged_df['Single Dual Bound Solution']) / merged_df['Best Known Solution']) * 100
).map('{:.2f}%'.format)

# --- 5. Format and Save ---
columns_order = [
    'Instance', 
    'Best Known Solution',
    'EA Dual Bound Solution', 'EA Dual Bound Optimality Gap', 'EA Dual Bound Expanded Nodes', 
    'EA Dual Bound Time', 'EA Dual Bound Optimality',
    'Single Dual Bound Solution', 'Single Dual Bound Optimality Gap', 'Single Dual Bound Expanded Nodes', 
    'Single Dual Bound Time', 'Single Dual Bound Optimality'
]
final_df = merged_df[columns_order]

# Save to CSV
final_df.to_csv(output_filename, index=False)

print(f"Aggregation complete. Results saved to {output_filename}")
print(final_df.head())

Aggregation complete. Results saved to aggregated_tsp_n20_results.csv
  Instance  Best Known Solution  EA Dual Bound Solution  \
0    1.txt                390.0                   390.0   
1   12.txt                399.0                   399.0   
2   18.txt                384.0                   384.0   
3   19.txt                400.0                   400.0   
4   21.txt                367.0                   367.0   

  EA Dual Bound Optimality Gap  EA Dual Bound Expanded Nodes  \
0                        0.00%                       3069154   
1                        0.00%                       3117551   
2                        0.00%                       4026838   
3                        0.00%                       3393691   
4                        0.00%                        456264   

   EA Dual Bound Time  EA Dual Bound Optimality  Single Dual Bound Solution  \
0         1908.576952                     False                       390.0   
1         1911.529572           